# PARCO for the PVRPWDP

Learning a Parallel AutoRegressive policy for the **Perishable Vehicle Routing Problem with Drones and Pickup (PVRPWDP)**.

**Key Features:**
- ⏰ Time windows constraints
- 🍎 Perishability (freshness) constraints  
- 🔋 Drone battery endurance limits
- 🚚🚁 Heterogeneous fleet (trucks + drones)

This notebook demonstrates training with **epoch-based data loading**.

In [1]:
%load_ext autoreload
%autoreload 2

import torch
from rl4co.utils.trainer import RL4COTrainer

from parco.envs.pvrpwdp import PVRPWDPVEnv
from parco.envs.pvrpwdp.generator import PVRPWDPGenerator
from parco.models import PARCORLModule, PARCOPolicy

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

e:\Đồ án\parco\.venv\Lib\site-packages\torchrl\data\replay_buffers\samplers.py:37: UserWarning: Failed to import torchrl C++ binaries. Some modules (eg, prioritized replay buffers) may not work with your installation. This is likely due to a discrepancy between your package version and the PyTorch version. Make sure both are compatible. Usually, torchrl majors follow the pytorch majors within a few days around the release. For instance, TorchRL 0.5 requires PyTorch 2.4.0, and TorchRL 0.6 requires PyTorch 2.5.0.
  warnings.warn(EXTENSION_WARNING)


Using device: cpu


e:\Đồ án\parco\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Environment Setup

We create the PVRPWDP environment with epoch-based data loading:
- **Training data**: Loaded from `train_data_npz/` (18 epoch files: epoch 0-17)
- **Validation data**: Loaded from `val_data/val.npz`
- **Test data**: Loaded from `test_data/test.npz`

If epoch files are missing, the environment will automatically fall back to the generator.

In [2]:
# Environment configuration
env = PVRPWDPVEnv(
    # Epoch data configuration (for training)
    epoch_data_dir="train_data_npz/",
    epoch_file_pattern="pvrpwdp_epoch_{epoch:02d}.npz",
    use_epoch_data=True,
    fallback_to_generator=True,
    
    # Standard validation/test files
    val_file="val_data/val.npz",
    test_file="test_data/test.npz",
    
    # Generator parameters (for fallback and val/test if files missing)
    generator_params=dict(
        num_loc=20,          # Number of customers
        num_agents=3,        # Number of vehicles (trucks + drones)
        min_demand=1,
        max_demand=10,
        capacity=40.0,       # Vehicle capacity
        endurance=10.0,      # Drone battery endurance
        speed=1.0,           # Truck speed
        drone_speed=2.0,     # Drone speed (faster than trucks)
        tw_expansion=3.0,    # Time window size factor
        freshness_factor=2.0, # Freshness constraint factor
    )
)

# Print epoch data information
env.print_epoch_data_info()

Epoch data directory 'train_data_npz/' does not exist. Epoch data loading will be disabled unless the directory is created.



EPOCH DATA CONFIGURATION
Epoch Data Directory: train_data_npz/
File Pattern:         pvrpwdp_epoch_{epoch:02d}.npz
Use Epoch Data:       True
Fallback to Generator: True
Current Epoch:        0
Max Epochs:           None

Available Epochs:     0



## Model Configuration

Configure the PARCO policy and RL module with PVRPWDP-specific settings.

Key configurations:
- **embed_dim**: 128 (embedding dimension)
- **num_augment**: 8 (SymNCO augmentations as baseline)
- **agent_handler**: "highprob" (select agent with highest probability)
- **Context embedding**: Includes time, capacity, endurance, and deadline features

In [3]:
emb_dim = 128

# Policy: Neural network for decision making
policy = PARCOPolicy(
    env_name=env.name,
    embed_dim=emb_dim,
    agent_handler="highprob",
    normalization="rms",
    context_embedding_kwargs={
        "normalization": "rms",
        "norm_after": False,
        "normalize_endurance_by_max": False,  # Normalize by time_scaler for consistency
        "use_time_to_depot": True,            # Include time-to-depot and time-to-deadline
    },
    norm_after=False,
)

# RL Module: Policy + Environment + Training algorithm
model = PARCORLModule(
    env, 
    policy=policy,
    train_data_size=1000,       # Instances per epoch (overridden by epoch files)
    val_data_size=100,          # Validation set size
    batch_size=64,              # Training batch size
    num_augment=8,              # SymNCO augmentations for shared baseline
    train_min_agents=3,         # Minimum number of agents
    train_max_agents=3,         # Maximum number of agents
    train_min_size=20,          # Minimum number of customers
    train_max_size=20,          # Maximum number of customers
    optimizer_kwargs={
        'lr': 1e-4,             # Learning rate
        'weight_decay': 0
    },
)

print("✅ Model created successfully!")
print(f"   Policy parameters: {sum(p.numel() for p in policy.parameters()):,}")

e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
e:\Đồ án\parco\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.


✅ Model created successfully!
   Policy parameters: 991,361


## Training

Train the model using RL4CO Trainer. 

**Note**: 
- Training data is loaded from epoch files in `train_data_npz/` (epochs 0-17)
- Validation data is loaded from `val_data/val.npz`
- Test data is loaded from `test_data/test.npz`

In [5]:
trainer = RL4COTrainer(
    max_epochs=18,           # Number of training epochs (matching available data)
    accelerator="gpu",       # Use "cpu" if no GPU available
    devices=1,               # Number of GPUs
    logger=None,             # Set to wandb/tensorboard for logging
    gradient_clip_val=1.0,   # Gradient clipping
)

print("🚀 Starting training...")
print(f"   Max epochs: {trainer.max_epochs}")
print(f"   Device: {trainer.accelerator}")

MisconfigurationException: No supported gpu backend found!

In [ ]:
# Start training
trainer.fit(model)

print("\n✅ Training completed!")

## Evaluate Trained Model

Now evaluate the trained model on the same test instances to see improvement.

### Greedy Decoding

Simply take the action with highest probability at each step.

In [ ]:
# Evaluate with greedy decoding
model = model.to(device)
out = model(td_init_test.clone(), phase="test", decode_type="greedy", return_actions=True)

actions = out["actions"]
print("\n🎯 Trained Model Performance (Greedy):")
print(f"   Average cost: {-out['reward'].mean().item():.2f}")

for i in range(3):
    cost = -out['reward'][i].item()
    print(f"\n   Instance {i+1} cost: {cost:.2f}")
    env.render(td_init[i], actions[i].cpu(), plot_number=True)

### Sampling Decoding

Sample N times from the probability distribution and take the best solution.

In [ ]:
# Evaluate with sampling (N=128 samples)
out_sampling = model(
    td_init_test.clone(), 
    phase="test", 
    decode_type="sampling",
    num_samples=128,
    return_actions=True
)

actions_sampling = out_sampling["actions"]
print("\n🎲 Trained Model Performance (Sampling x128):")
print(f"   Average cost: {-out_sampling['reward'].mean().item():.2f}")

for i in range(3):
    cost = -out_sampling['reward'][i].item()
    print(f"\n   Instance {i+1} cost: {cost:.2f}")
    env.render(td_init[i], actions_sampling[i].cpu(), plot_number=True)

### Augmentation Decoding

Augment the state N times (8 geometric transformations) and take the best solution.

In [ ]:
# Evaluate with augmentation
out_augment = model(
    td_init_test.clone(), 
    phase="test", 
    decode_type="greedy",
    augment=True,
    num_augment=8,
    return_actions=True
)

actions_augment = out_augment["actions"]
print("\n🔄 Trained Model Performance (Augmentation x8):")
print(f"   Average cost: {-out_augment['reward'].mean().item():.2f}")

for i in range(3):
    cost = -out_augment['reward'][i].item()
    print(f"\n   Instance {i+1} cost: {cost:.2f}")
    env.render(td_init[i], actions_augment[i].cpu(), plot_number=True)

## Performance Summary

Compare all decoding methods.

In [ ]:
import pandas as pd

# Create comparison table
results = {
    'Method': ['Untrained', 'Greedy', 'Sampling (x128)', 'Augmentation (x8)'],
    'Average Cost': [
        -out['reward'].mean().item(),  # From untrained cell
        -out['reward'].mean().item(),  # Greedy
        -out_sampling['reward'].mean().item(),  # Sampling
        -out_augment['reward'].mean().item(),   # Augmentation
    ]
}

df = pd.DataFrame(results)
print("\n📊 Performance Comparison:")
print(df.to_string(index=False))

# Calculate improvement
improvement = (results['Average Cost'][0] - results['Average Cost'][3]) / results['Average Cost'][0] * 100
print(f"\n💡 Improvement: {improvement:.1f}% (Untrained → Aug x8)")

## Epoch Data Management

Useful utilities for managing epoch training data.

### List Available Epochs

In [ ]:
# List available epoch files
available_epochs = env.list_available_epochs()
print(f"📂 Available epoch files: {len(available_epochs)}")
if available_epochs:
    print(f"   Epoch range: {min(available_epochs)} - {max(available_epochs)}")
    if len(available_epochs) <= 20:
        print(f"   Epochs: {available_epochs}")
    else:
        print(f"   Sample: {available_epochs[:10]} ... {available_epochs[-10:]}")
else:
    print("   ⚠️  No epoch files found. Using generator fallback.")

### Validate Epoch Files

In [ ]:
# Validate epoch files (check if loadable)
validation_results = env.validate_epoch_files(max_epoch=trainer.max_epochs)

print("\n🔍 Epoch File Validation:")
print(f"   Total expected: {validation_results['total_expected']}")
print(f"   ✅ Valid:       {len(validation_results['valid'])}")
print(f"   ❌ Missing:     {len(validation_results['missing'])}")
print(f"   ⚠️  Corrupted:  {len(validation_results['corrupted'])}")

if validation_results['missing']:
    print(f"\n   Missing epochs: {validation_results['missing'][:20]}")
if validation_results['corrupted']:
    print(f"   Corrupted epochs: {validation_results['corrupted']}")

## Next Steps

### Generate Epoch Data

If you don't have epoch files yet, generate them using:

```bash
# Fixed difficulty (for testing)
python generate_pvrpwdp_epochs.py \
    --num_epochs 10 \
    --batch_size 1000 \
    --num_loc 20 \
    --num_agents 3 \
    --output_dir data/pvrpwdp/train_epochs/

# Curriculum learning (for production training)
python generate_pvrpwdp_epochs.py \
    --curriculum \
    --num_epochs 100 \
    --batch_size 1000 \
    --start_num_loc 10 \
    --end_num_loc 50 \
    --num_agents 3 \
    --output_dir data/pvrpwdp/train_epochs/
```

### Generate Validation/Test Data

```bash
# Generate validation set
python -c "
from parco.envs.pvrpwdp import PVRPWDPVEnv
env = PVRPWDPVEnv(generator_params={'num_loc': 20, 'num_agents': 3})
td = env.generator(batch_size=[100])
from rl4co.data.utils import save_tensordict_to_npz
save_tensordict_to_npz(td, 'data/pvrpwdp/val.npz')
print('✅ Saved val.npz')
"

# Generate test set
python -c "
from parco.envs.pvrpwdp import PVRPWDPVEnv
env = PVRPWDPVEnv(generator_params={'num_loc': 20, 'num_agents': 3})
td = env.generator(batch_size=[100])
from rl4co.data.utils import save_tensordict_to_npz
save_tensordict_to_npz(td, 'data/pvrpwdp/test.npz')
print('✅ Saved test.npz')
"
```

### Hyperparameter Tuning

Key hyperparameters to tune:
- `embed_dim`: 128, 256, 512
- `num_augment`: 4, 8, 16
- `learning_rate`: 1e-4, 5e-5, 1e-5
- `batch_size`: 32, 64, 128
- `normalize_endurance_by_max`: True/False

### Curriculum Learning

Enable curriculum learning by generating epoch files with increasing difficulty:
- Epoch 0-20: 10-15 customers (easy)
- Epoch 20-50: 15-30 customers (medium)
- Epoch 50-100: 30-50 customers (hard)

This helps the model learn progressively more complex routing patterns.


## Troubleshooting

### Issue: Epoch files not loading

**Solution**: Check if files exist and are valid:
```python
env.print_epoch_data_info()
env.validate_epoch_files(max_epoch=10)
```

### Issue: Out of memory

**Solutions**:
- Reduce `batch_size`: 64 → 32
- Reduce `embed_dim`: 128 → 64
- Reduce `num_augment`: 8 → 4
- Use gradient accumulation

### Issue: Model not learning

**Solutions**:
- Check data quality (all customers visited?)
- Increase training epochs
- Adjust learning rate
- Try different `agent_handler` strategies
- Enable logging (wandb/tensorboard)

### Issue: Slow training

**Solutions**:
- Use pre-generated epoch files (avoid on-the-fly generation)
- Increase `batch_size` (better GPU utilization)
- Use multiple GPUs: `devices=2`
- Profile with `profiler="simple"`
